Loading Data

In [10]:
import pandas as pd
import numpy as np

In [11]:
results = pd.read_csv('../data/raw/results.csv', parse_dates = ['date'])
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [12]:
rankings = pd.read_csv('../data/raw/fifa_ranking.csv', parse_dates = ['rank_date'])
rankings.head()

,Unnamed: 0,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [13]:
results.isnull().sum()

date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [14]:
rankings.isnull().sum()

Unnamed: 0         0
rank               9
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64

In [15]:
results_modern = results[results['date'].dt.year>= 2006].copy()
results_modern.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
29727,2006-01-02,Qatar,Libya,2.0,0.0,Friendly,Doha,Qatar,False
29728,2006-01-05,Egypt,Zimbabwe,2.0,0.0,Friendly,Alexandria,Egypt,False
29729,2006-01-07,Guinea,Togo,1.0,0.0,Friendly,Viry-Châtillon,France,True
29730,2006-01-09,Morocco,DR Congo,3.0,0.0,Friendly,Rabat,Morocco,False
29731,2006-01-11,Ghana,Togo,0.0,1.0,Friendly,Monastir,Tunisia,True


In [16]:
rankings_modern = rankings[rankings['rank_date'] >= '2005-06-01'].copy()
rankings_modern.head()

,Unnamed: 0,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
25532,25532,65.0,Hungary,HUN,561.0,552.0,-4,UEFA,2005-06-15
25533,25533,74.0,Austria,AUT,548.0,547.0,2,UEFA,2005-06-15
25534,25534,73.0,Zambia,ZAM,550.0,554.0,5,CAF,2005-06-15
25535,25535,72.0,Angola,ANG,551.0,545.0,-1,CAF,2005-06-15
25536,25536,70.0,Cuba,CUB,554.0,563.0,6,CONCACAF,2005-06-15


In [17]:
rankings_modern = rankings_modern.rename(columns = {'country_full': 'team', 'rank': 'fifa_rank', 'rank_date': 'date'})
results_modern = results_modern.sort_values('date')
rankings_modern = rankings_modern.sort_values('date')

Merging both datasets

In [18]:
name_cleaner = {
    'IR Iran': 'Iran',
    'Korea Republic': 'South Korea',
    'Czechia': 'Czech Republic',
    'USA': 'United States',
    "Côte d'Ivoire": 'Ivory Coast',
    'Congo DR': 'DR Congo',
    'Cabo Verde': 'Cape Verde',
    'Curaçao': 'Curacao',
    'Türkiye': 'Turkey'
}
rankings_modern['team'] = rankings_modern['team'].replace(name_cleaner)
results_modern['home_team'] = results_modern['home_team'].replace({'Curaçao': 'Curacao'})
results_modern['away_team'] = results_modern['away_team'].replace({'Curaçao': 'Curacao'})

In [19]:
results_modern = pd.merge_asof(
    results_modern,
    rankings_modern[['date', 'team', 'fifa_rank']],
    on = 'date',
    left_by = 'home_team',
    right_by = 'team', 
    direction = 'backward'
).rename(columns = {'fifa_rank': 'home_rank'})

In [20]:
results_modern.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,team,home_rank
0,2006-01-02,Qatar,Libya,2.0,0.0,Friendly,Doha,Qatar,False,Qatar,95.0
1,2006-01-05,Egypt,Zimbabwe,2.0,0.0,Friendly,Alexandria,Egypt,False,Egypt,32.0
2,2006-01-07,Guinea,Togo,1.0,0.0,Friendly,Viry-Châtillon,France,True,Guinea,79.0
3,2006-01-09,Morocco,DR Congo,3.0,0.0,Friendly,Rabat,Morocco,False,Morocco,36.0
4,2006-01-11,Ghana,Togo,0.0,1.0,Friendly,Monastir,Tunisia,True,Ghana,50.0


In [12]:
results_modern = pd.merge_asof(
    results_modern, 
    rankings_modern[['date', 'team', 'fifa_rank']],
    on = 'date', 
    left_by = 'away_team', 
    right_by = 'team', 
    direction = 'backward'
).rename(columns = {'fifa_rank': 'away_rank'}).drop(columns = ['team'], errors = 'ignore')

In [13]:
results_modern.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rank,away_rank
0,2006-01-02,Qatar,Libya,2.0,0.0,Friendly,Doha,Qatar,False,95.0,80.0
1,2006-01-05,Egypt,Zimbabwe,2.0,0.0,Friendly,Alexandria,Egypt,False,32.0,53.0
2,2006-01-07,Guinea,Togo,1.0,0.0,Friendly,Viry-Châtillon,France,True,79.0,56.0
3,2006-01-09,Morocco,DR Congo,3.0,0.0,Friendly,Rabat,Morocco,False,36.0,77.0
4,2006-01-11,Ghana,Togo,0.0,1.0,Friendly,Monastir,Tunisia,True,50.0,56.0


In [14]:
results_modern.isnull().sum()

date             0
home_team        0
away_team        0
home_score      72
away_score      72
tournament       0
city             0
country          0
neutral          0
home_rank     1488
away_rank     1612
dtype: int64

In [15]:
unmapped_teams = results_modern[results_modern['home_rank'].isnull()]['home_team'].unique()
print("Teams with missing ranks:", unmapped_teams)

Teams with missing ranks: ['Northern Cyprus' 'Hong Kong' 'Monaco' 'Saint Kitts and Nevis' 'Alderney'
 'Kyrgyzstan' 'Brunei' 'Kosovo' 'Jersey' 'Guadeloupe' 'Basque Country'
 'Catalonia' 'Republic of St. Pauli' 'Zanzibar' 'Gibraltar' 'Greenland'
 'Gambia' 'Curacao' 'Martinique' 'United States Virgin Islands'
 'Saint Lucia' 'Saint Vincent and the Grenadines' 'Timor-Leste' 'Găgăuzia'
 'Tibet' 'Occitania' 'Sápmi' 'Silesia' 'Réunion' 'Galicia' 'Montenegro'
 'Kernow' 'Northern Mariana Islands' 'Guernsey' 'Romani people'
 'North Korea' 'Rhodes' 'Menorca' 'Åland Islands' 'Western Isles'
 'Ynys Môn' 'Gotland' 'Saare County' 'Frøya' 'Tuvalu' 'Taiwan'
 'Saint Martin' 'French Guiana' 'Andalusia' 'Canary Islands' 'Brittany'
 'Provence' 'Arameans Suryoye' 'Padania' 'Iraqi Kurdistan' 'Mayotte'
 'Corsica' 'Falkland Islands' 'Shetland' 'Isle of Man' 'Isle of Wight'
 'Gozo' 'Kiribati' 'Bonaire' 'Chagos Islands' 'Sealand' 'Western Sahara'
 'Raetia' 'Darfur' 'Tamil Eelam' 'South Sudan' 'Saint Barthélemy'
 

In [16]:
results_modern['home_rank'] = results_modern['home_rank'].fillna(150)
results_modern['away_rank'] = results_modern['away_rank'].fillna(150)
results_modern.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19560 entries, 0 to 19559
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        19560 non-null  datetime64[ns]
 1   home_team   19560 non-null  object        
 2   away_team   19560 non-null  object        
 3   home_score  19488 non-null  float64       
 4   away_score  19488 non-null  float64       
 5   tournament  19560 non-null  object        
 6   city        19560 non-null  object        
 7   country     19560 non-null  object        
 8   neutral     19560 non-null  bool          
 9   home_rank   19560 non-null  float64       
 10  away_rank   19560 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(4), object(5)
memory usage: 1.5+ MB


In [17]:
results_modern['rank_diff'] = results_modern['home_rank'] - results_modern['away_rank']
results_modern.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rank,away_rank,rank_diff
0,2006-01-02,Qatar,Libya,2.0,0.0,Friendly,Doha,Qatar,False,95.0,80.0,15.0
1,2006-01-05,Egypt,Zimbabwe,2.0,0.0,Friendly,Alexandria,Egypt,False,32.0,53.0,-21.0
2,2006-01-07,Guinea,Togo,1.0,0.0,Friendly,Viry-Châtillon,France,True,79.0,56.0,23.0
3,2006-01-09,Morocco,DR Congo,3.0,0.0,Friendly,Rabat,Morocco,False,36.0,77.0,-41.0
4,2006-01-11,Ghana,Togo,0.0,1.0,Friendly,Monastir,Tunisia,True,50.0,56.0,-6.0


Calculating rolling form

In [18]:
df_home = results_modern[['date', 'home_team', 'home_score', 'away_score']].copy().rename(
    columns = {'home_team': 'team', 'home_score': 'goals_scored', 'away_score': 'goals_conceded'}
)
df_home['is_home'] = 1
df_home.head()

,date,team,goals_scored,goals_conceded,is_home
0,2006-01-02,Qatar,2.0,0.0,1
1,2006-01-05,Egypt,2.0,0.0,1
2,2006-01-07,Guinea,1.0,0.0,1
3,2006-01-09,Morocco,3.0,0.0,1
4,2006-01-11,Ghana,0.0,1.0,1


In [19]:
df_away = results_modern[['date', 'away_team', 'away_score', 'home_score']].copy().rename(
    columns = {'away_team': 'team', 'away_score': 'goals_scored', 'home_score': 'goals_conceded'}
)
df_away['is_home'] = 0
df_away.head()

,date,team,goals_scored,goals_conceded,is_home
0,2006-01-02,Libya,0.0,2.0,0
1,2006-01-05,Zimbabwe,0.0,2.0,0
2,2006-01-07,Togo,0.0,1.0,0
3,2006-01-09,DR Congo,0.0,3.0,0
4,2006-01-11,Togo,1.0,0.0,0


In [20]:
team_timeline = pd.concat([df_home, df_away]).sort_values(['team', 'date'])
team_timeline['is_win'] = (team_timeline['goals_scored'] > team_timeline['goals_conceded']).astype(int)

In [21]:
team_timeline['rolling_goals_scored'] = team_timeline.groupby('team')['goals_scored'].transform(
    lambda x: x.shift(1).rolling(window = 5, min_periods = 1).mean()
)
team_timeline['rolling_goals_conceded'] = team_timeline.groupby('team')['goals_conceded'].transform(
    lambda x: x.shift(1).rolling(window = 5, min_periods = 1).mean()
)
team_timeline['rolling_win_rate'] = team_timeline.groupby('team')['is_win'].transform(
    lambda x: x.shift(1).rolling(window = 5, min_periods = 1).mean()
)

In [22]:
features_to_merge = ['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded', 'rolling_win_rate']

In [23]:
home_stats = team_timeline[team_timeline['is_home'] == 1][features_to_merge].copy()
home_stats = home_stats.rename(columns =
    {'team': 'home_team', 'rolling_goals_scored': 'home_rolling_goals_scored',
     'rolling_goals_conceded': 'home_rolling_goals_conceded', 'rolling_win_rate': 'home_rolling_win_rate'
    })
away_stats = team_timeline[team_timeline['is_home'] == 0][features_to_merge].copy()
away_stats = away_stats.rename(columns =
    {'team': 'away_team', 'rolling_goals_scored': 'away_rolling_goals_scored',
     'rolling_goals_conceded': 'away_rolling_goals_conceded', 'rolling_win_rate': 'away_rolling_win_rate'
    })

In [24]:
results_modern = pd.merge(results_modern, home_stats, on = ['date', 'home_team'], how = 'left')
results_modern = pd.merge(results_modern, away_stats, on = ['date', 'away_team'], how = 'left')

In [25]:
results_modern.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rank,away_rank,rank_diff,home_rolling_goals_scored,home_rolling_goals_conceded,home_rolling_win_rate,away_rolling_goals_scored,away_rolling_goals_conceded,away_rolling_win_rate
17434,2024-06-04,Austria,Serbia,2.0,1.0,Friendly,Vienna,Austria,False,25.0,33.0,-8.0,2.6,0.2,1.0,1.4,1.2,0.4
15770,2022-09-24,Scotland,Republic of Ireland,2.0,1.0,UEFA Nations League,Glasgow,Scotland,False,45.0,47.0,-2.0,2.0,1.4,0.6,1.0,0.6,0.4
16885,2023-10-17,Guam,Singapore,0.0,1.0,FIFA World Cup qualification,Dededo,Guam,False,201.0,157.0,44.0,0.4,2.2,0.0,1.6,1.4,0.4
2192,2008-06-02,Thailand,Bahrain,2.0,3.0,FIFA World Cup qualification,Bangkok,Thailand,False,96.0,72.0,24.0,2.6,1.8,0.4,1.2,0.2,1.0
11263,2017-10-05,Venezuela,Uruguay,0.0,0.0,FIFA World Cup qualification,San Cristóbal,Venezuela,False,68.0,16.0,52.0,0.8,1.2,0.0,0.8,1.8,0.2


In [26]:
avg_home_scored = results_modern['home_score'].dropna().mean()
avg_away_scored = results_modern['away_score'].dropna().mean()

In [27]:
#historical averages and baseline averages
results_modern['home_rolling_win_rate'] = results_modern['home_rolling_win_rate'].fillna(0.33)
results_modern['away_rolling_win_rate'] = results_modern['away_rolling_win_rate'].fillna(0.33)
results_modern['home_rolling_goals_scored'] = results_modern['home_rolling_goals_scored'].fillna(avg_home_scored)
results_modern['away_rolling_goals_scored'] = results_modern['home_rolling_win_rate'].fillna(avg_away_scored)
results_modern['home_rolling_goals_conceded'] = results_modern['home_rolling_goals_conceded'].fillna(avg_away_scored)
results_modern['away_rolling_goals_conceded'] = results_modern['away_rolling_goals_conceded'].fillna(avg_home_scored)

In [28]:
results_modern.isnull().sum()

date                            0
home_team                       0
away_team                       0
home_score                     72
away_score                     72
tournament                      0
city                            0
country                         0
neutral                         0
home_rank                       0
away_rank                       0
rank_diff                       0
home_rolling_goals_scored       0
home_rolling_goals_conceded     0
home_rolling_win_rate           0
away_rolling_goals_scored       0
away_rolling_goals_conceded     0
away_rolling_win_rate           0
dtype: int64

In [29]:
results_modern = results_modern.drop(columns = ['city', 'country'], errors = 'ignore')
results_modern.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_rank,away_rank,rank_diff,home_rolling_goals_scored,home_rolling_goals_conceded,home_rolling_win_rate,away_rolling_goals_scored,away_rolling_goals_conceded,away_rolling_win_rate
11912,2018-06-24,Japan,Senegal,2.0,2.0,FIFA World Cup,True,61.0,27.0,34.0,1.40000,1.800000,0.40,0.40,0.6,0.4
184,2006-04-22,Kosovo,Monaco,7.0,1.0,Friendly,True,150.0,150.0,0.0,1.60874,1.111561,0.33,0.33,1.0,1.0
237,2006-05-27,Romania,Colombia,0.0,0.0,Friendly,True,25.0,27.0,-2.0,1.50000,0.500000,0.75,0.75,1.0,0.0
12850,2019-06-15,Indonesia,Vanuatu,6.0,0.0,Friendly,False,160.0,163.0,-3.0,1.60000,1.800000,0.40,0.40,1.2,0.2
1050,2007-03-28,Malta,Greece,0.0,1.0,UEFA Euro qualification,False,111.0,13.0,98.0,1.00000,1.800000,0.20,0.20,1.2,0.4


Defining weighted importance 

In [30]:
tournament_weights = {
    'FIFA World Cup': 4.0,
    'UEFA Euro': 3.0,
    'Copa América': 3.0,
    'African Cup of Nations': 3.0,
    'FIFA World Cup qualification': 2.5,
    'Friendly': 1.0
}
results_modern['tournament_weight'] = results_modern['tournament'].map(tournament_weights).fillna(1.5)
target_date = results_modern['date'].max()
results_modern['days_since'] = (target_date - results_modern['date']).dt.days
results_modern['recency_weight'] = np.exp(-results_modern['days_since'] / (4*365 + 1))
results_modern['match_weight']  = results_modern['tournament_weight'] * results_modern['recency_weight']

In [31]:
results_modern.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_rank,away_rank,rank_diff,home_rolling_goals_scored,home_rolling_goals_conceded,home_rolling_win_rate,away_rolling_goals_scored,away_rolling_goals_conceded,away_rolling_win_rate,tournament_weight,days_since,recency_weight,match_weight
19561,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,True,17.0,63.0,-46.0,1.666667,1.333333,0.4,0.4,1.333333,0.2,4.0,1,0.999316,3.997263
9093,2015-06-16,Myanmar,South Korea,0.0,2.0,FIFA World Cup qualification,True,143.0,58.0,85.0,1.000000,2.200000,0.0,0.0,0.600000,0.6,2.5,4029,0.063437,0.158594
3304,2009-06-10,England,Andorra,6.0,0.0,FIFA World Cup qualification,False,6.0,196.0,-190.0,2.400000,0.800000,0.8,0.8,3.400000,0.0,2.5,6226,0.014102,0.035254
16952,2023-11-17,Denmark,Slovenia,2.0,1.0,UEFA Euro qualification,False,19.0,54.0,-35.0,2.200000,0.600000,0.8,0.8,0.600000,0.8,1.5,953,0.520850,0.781275
7688,2013-11-15,Turkey,Northern Ireland,1.0,0.0,Friendly,False,40.0,90.0,-50.0,2.200000,0.800000,0.6,0.6,2.000000,0.2,1.0,4607,0.042710,0.042710


In [32]:
results_modern = results_modern.drop(columns = ['tournament', 'tournament_weight', 'recency_weight'], errors = 'ignore')
results_modern.sample(5)

,date,home_team,away_team,home_score,away_score,neutral,home_rank,away_rank,rank_diff,home_rolling_goals_scored,home_rolling_goals_conceded,home_rolling_win_rate,away_rolling_goals_scored,away_rolling_goals_conceded,away_rolling_win_rate,days_since,match_weight
33,2006-01-25,Saudi Arabia,Greece,1.0,1.0,False,33.0,16.0,17.0,1.0,1.0,0.0,0.0,1.0,0.0,7458,0.006068
10132,2016-06-15,France,Albania,2.0,0.0,False,17.0,42.0,-25.0,3.0,1.4,1.0,1.0,1.4,0.4,3664,0.244324
18652,2025-06-10,San Marino,Austria,0.0,4.0,False,210.0,22.0,188.0,1.0,2.0,0.2,0.2,1.0,0.4,382,1.924808
12264,2018-10-16,Cambodia,Singapore,1.0,2.0,False,169.0,166.0,3.0,1.0,1.6,0.2,0.2,0.8,0.6,2811,0.146018
17223,2024-01-29,Ivory Coast,Senegal,1.0,1.0,False,49.0,20.0,29.0,1.8,1.2,0.6,0.6,0.2,0.8,880,1.642608


In [33]:
results_modern['home_attack_strength'] = results_modern['home_rolling_goals_scored'] / avg_home_scored
results_modern['away_defense_weakness'] = results_modern['away_rolling_goals_conceded'] / avg_home_scored
results_modern['exp_home_goals'] = results_modern['home_attack_strength'] * results_modern['away_defense_weakness'] * avg_home_scored
results_modern['away_attack_strength'] = results_modern['away_rolling_goals_scored'] / avg_away_scored
results_modern['home_defense_weakness'] = results_modern['home_rolling_goals_conceded'] / avg_away_scored
results_modern['exp_away_goals'] = results_modern['away_attack_strength'] * results_modern['home_defense_weakness'] * avg_away_scored
results_modern.sample(5)

,date,home_team,away_team,home_score,away_score,neutral,home_rank,away_rank,rank_diff,home_rolling_goals_scored,...,away_rolling_goals_conceded,away_rolling_win_rate,days_since,match_weight,home_attack_strength,away_defense_weakness,exp_home_goals,away_attack_strength,home_defense_weakness,exp_away_goals
5387,2011-09-02,Republic of Ireland,Slovakia,0.0,0.0,False,31.0,26.0,5.0,2.0,...,1.0,0.6,5412,0.036926,1.243209,0.621604,1.243209,0.719708,0.000000,0.000000
2292,2008-06-14,Greece,Russia,0.0,1.0,True,8.0,24.0,-16.0,1.2,...,1.8,0.6,6587,0.033043,0.745925,1.118888,1.342665,0.359854,1.079563,0.431825
7207,2013-06-07,Liechtenstein,Slovakia,1.0,1.0,False,158.0,56.0,102.0,0.2,...,1.4,0.0,4768,0.095634,0.124321,0.870246,0.174049,0.000000,1.259490,0.000000
4196,2010-07-01,Tunisia,Botswana,0.0,1.0,False,55.0,116.0,-61.0,2.0,...,0.2,0.6,5840,0.027549,1.243209,0.124321,0.248642,0.179927,1.079563,0.215913
14780,2021-10-09,Kazakhstan,Bosnia and Herzegovina,0.0,2.0,False,120.0,57.0,63.0,1.0,...,1.0,0.2,1722,0.769239,0.621604,0.621604,0.621604,0.000000,1.799271,0.000000


In [34]:
results_modern.isnull().sum()

date                            0
home_team                       0
away_team                       0
home_score                     72
away_score                     72
neutral                         0
home_rank                       0
away_rank                       0
rank_diff                       0
home_rolling_goals_scored       0
home_rolling_goals_conceded     0
home_rolling_win_rate           0
away_rolling_goals_scored       0
away_rolling_goals_conceded     0
away_rolling_win_rate           0
days_since                      0
match_weight                    0
home_attack_strength            0
away_defense_weakness           0
exp_home_goals                  0
away_attack_strength            0
home_defense_weakness           0
exp_away_goals                  0
dtype: int64

In [36]:
#make feature_eng.py using this .ipynb and adding the shootout override code and keeping all necessary feature eng steps that i might have missed